# Day 1 · Module 2 — Computer Vision for Radiology (Knowing When You Don't Know)

**Objective:** train an image classifier, then read its **confidence**. A high softmax is not knowledge, so a safe model must sometimes **refuse to answer**.

> **Instructor solution notebook** · kernel **AImed**.

### References
- MedMNIST `PneumoniaMNIST`; softmax confidence; out-of-distribution inputs.
- *Today's Goal:* if a model is 99% confident on an upside-down X-ray, what does its 'confidence' actually measure?

In [ ]:
# --- Environment setup (run me first; you don't need to edit this) ---
# Finds the shared course 'scripts/' folder and puts it on Python's import path so the
# later cells can do `import clinical_risk_console` and `from common import metrics`.
import sys, pathlib, os
cands = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]   # current dir + all parent dirs
if os.environ.get('PROJECT_DIR'):                            # + the shared course dir if it is set
    cands.append(pathlib.Path(os.environ['PROJECT_DIR']))
scripts = next((b/'scripts' for b in cands                   # first candidate that has scripts/common/
                if (b/'scripts'/'common'/'aimed_config.py').exists()), None)
# if this assert fires, your kernel isn't seeing the course tree -> relaunch the AImed kernel
assert scripts, 'run from inside the course tree, or set PROJECT_DIR to the shared course dir'
sys.path.insert(0, str(scripts))                             # make those scripts importable below
from common import aimed_config as cfg                       # shared paths + cache configuration
cfg.setup_caches()                                           # point HF/torch caches at shared storage
print('environment OK  ->', cfg.PROJECT_DIR)

## Guided hands-on

### Background — confidence, out-of-distribution inputs, and abstention

**A neural network's softmax is not a measure of how much it knows.**

**The model.** We fine-tune a small image classifier (a **ResNet-18 CNN**) on chest X-rays to answer *pneumonia, or not?* Its softmax outputs two numbers that sum to 1; the larger one is its **confidence** (0.5 = a coin flip, 1.0 = 'certain').

**The catch — overconfidence on the unfamiliar.** Deep ReLU networks are provably pushed to **high** confidence on inputs far from their training data (Hein et al. 2019). So a pneumonia model will hand you a 99%-confident 'diagnosis' for pure noise, an upside-down scan, or a totally different image — it was never taught to say *'this isn't even a chest X-ray.'*

**Out-of-distribution (OOD).** Anything unlike the training data: a different scanner or modality, a corrupted image, a body part the model never saw. In a real clinic these arrive constantly.

**Abstention / selective prediction.** The remedy is to let the model **refuse** — answer only when some signal clears a bar, else output 'I don't know' and route to a human. You trade **coverage** (the fraction you answer) for **accuracy** (on the ones you do). This module asks two things: (1) does abstaining on the least-confident cases raise accuracy on the rest? and (2) can a confidence bar alone catch the OOD garbage? It ties straight back to Module 1: *calibration asked whether the probabilities are honest; abstention is what you DO when they are not.*

In [ ]:
# Guided hands-on — train ONE pneumonia model, then ask it about inputs it never trained on.
import radiograph_reader as RR
import numpy as np, os
try:                                     # real PneumoniaMNIST when cached...
    Xtr, ytr = RR.load_pneumonia('train', size=64); Xte, yte = RR.load_pneumonia('test', size=64)
    EPOCHS, PRE = 3, True
except Exception as e:                    # ...else the offline smoke fixture
    print('MedMNIST not cached, using smoke fixture:', e)
    Xtr, ytr = RR.load_pneumonia(smoke=True); Xte, yte = Xtr, ytr; EPOCHS, PRE = 2, False
model = RR.train(RR.build_model(pretrained=PRE), Xtr, ytr, epochs=EPOCHS, log=lambda *a: None)

def confidence(X):                        # max-softmax confidence of the 2-class model
    p = RR.predict_proba(model, X)         # P(pneumonia)
    return np.maximum(p, 1 - p)

# in-distribution: how good is it, and how confident is it when it answers?
p_id = RR.predict_proba(model, Xte); acc_id = float(((p_id >= 0.5).astype(int) == yte).mean())
print(f'IN-DISTRIBUTION (real chest X-rays): accuracy {acc_id:.2f} | AUROC {RR.evaluate(model, Xte, yte):.2f} | mean confidence {confidence(Xte).mean():.2f}')

# now show it things it should REFUSE — inputs from outside its training world
rng = np.random.default_rng(0)
ood = {'pure noise': rng.random((400, 1, 64, 64), dtype='float32'),
       'upside-down X-ray': 1.0 - Xte[:400]}
try:                                       # cached chestmnist = other CXR findings it never trained on
    z = np.load(os.path.join(os.environ['MEDMNIST_ROOT'], 'chestmnist_64.npz'))
    xc = (z['test_images'][:400].astype('float32') / 255.0)
    ood['other-finding CXR'] = xc[:, None] if xc.ndim == 3 else xc.transpose(0, 3, 1, 2)
except Exception:
    pass
print('OUT-OF-DISTRIBUTION (the model has no business judging these):')
for name, Xo in ood.items():
    c = confidence(Xo)
    print(f'  {name:20s} mean confidence {c.mean():.2f} | {100 * (c > 0.8).mean():.0f}% are >80% sure')
print('--> the model is often as confident on these out-of-distribution inputs as on real X-rays. A high softmax is NOT knowing.')

## Your turn — when may the model answer? Build the tradeoff.

**Predict first (no code).** Is there a single confidence bar τ that keeps accuracy high *and* blocks the OOD noise? Jot down yes/no and one sentence why.

**Now find out.** Using the toolbox below: first tabulate the model's confidence on real vs out-of-distribution inputs, then sweep the bar τ and watch coverage, accuracy, and how much OOD noise still leaks through.

In [ ]:
# WORKED — run the model on each kind of input and tabulate its confidence.
import numpy as np, pandas as pd
inputs = {'real X-rays (ID)': Xte, **ood}     # in-distribution + every OOD set from the guided cell
rows = []
for name, Xset in inputs.items():
    c = confidence(Xset)
    rows.append({
        'input':           name,
        'mean_confidence': round(float(c.mean()), 2),
        'pct_over_80':     round(100 * float((c > 0.8).mean()), 0),
    })
confidence_table = pd.DataFrame(rows)
print(confidence_table.to_string(index=False))
print('Sanity check: these match the numbers the guided cell printed above.')

In [ ]:
# WORKED — sweep the confidence bar tau: one row per bar.
import numpy as np, pandas as pd
conf_id = confidence(Xte)
correct = (RR.predict_proba(model, Xte) >= 0.5).astype(int) == yte
conf_noise = confidence(np.random.default_rng(3).random((400, 1, 64, 64), dtype='float32'))
rows = []
for tau in np.round(np.arange(0.5, 1.0, 0.05), 2):
    keep = conf_id >= tau
    rows.append({
        'tau': tau,
        'coverage':             round(float(keep.mean()), 2),
        'accuracy_on_answered': (round(float(correct[keep].mean()), 2) if keep.any() else float('nan')),
        'noise_leakage':        round(float((conf_noise >= tau).mean()), 2),
    })
abstain_table = pd.DataFrame(rows)
print(abstain_table.to_string(index=False))
print('Note: raising tau trades coverage for accuracy on the answered -- but noise_leakage shows a confidence bar does not reliably fence OOD; a dedicated novelty detector is the real guard.')

In [ ]:
# Stuck building it? This makes the same table so you can keep going.
import numpy as np, pandas as pd
conf_id = confidence(Xte)
correct = (RR.predict_proba(model, Xte) >= 0.5).astype(int) == yte
conf_noise = confidence(np.random.default_rng(3).random((400, 1, 64, 64), dtype='float32'))
abstain_table = pd.DataFrame([
    {'tau': tau,
     'coverage': round(float((conf_id >= tau).mean()), 2),
     'accuracy_on_answered': (round(float(correct[conf_id >= tau].mean()), 2) if (conf_id >= tau).any() else float('nan')),
     'noise_leakage': round(float((conf_noise >= tau).mean()), 2)}
    for tau in np.round(np.arange(0.5, 1.0, 0.05), 2)])
print(abstain_table.to_string(index=False))

### Don't feel like building the table? Pick one bar and read it
Fill in `tau` below and read coverage / accuracy / noise-leakage at that single confidence bar.

In [ ]:
# A worked single bar (yours may differ):
import numpy as np
conf_id = confidence(Xte)
correct = (RR.predict_proba(model, Xte) >= 0.5).astype(int) == yte
noise = np.random.default_rng(3).random((400, 1, 64, 64), dtype='float32')
tau = 0.90
keep = conf_id >= tau
print(f'tau={tau}: coverage {keep.mean():.2f} | accuracy-on-answered {correct[keep].mean():.2f} | '
      f'noise that still passes {(confidence(noise) >= tau).mean():.2f}')

### Your task — decide when the model may answer
Using the numbers above, fill in the **`DECISION`** cell: what confidence bar would you ship for a triage tool, and what does it cost you in **coverage** (how many patients you punt to a human)? Does that bar actually stop the model from confidently mislabeling **out-of-distribution** inputs (noise, an upside-down or non-chest image)? If not, what *should* gate the model instead? What matters is the reasoning, argued in the discussion.

In [ ]:
# A defensible answer (yours may differ — the REASONING is what matters):
DECISION = {
    'ship_threshold': 0.90,
    'coverage_cost': 'we punt the least-confident slice of patients to a radiologist',
    'catches_ood': False,
    'real_guardrail': ('A confidence bar only trims borderline in-distribution cases; OOD inputs are '
                       'confidently WRONG, so you need an explicit out-of-distribution / novelty '
                       'detector (e.g. feature-space distance) plus input sanity checks BEFORE the model.'),
}
print(DECISION)

### Checkpoint — stuck? See why confidence isn't safety
Couldn't decide on a bar? Run the next cell (it reuses the model you trained above): it plots the model's confidence on real vs out-of-distribution inputs and the coverage-accuracy tradeoff, and the caption tells you how to read it.

In [ ]:
# CHECKPOINT — stuck? This VISUALIZES why a high softmax is not safety.
%matplotlib inline
import matplotlib.pyplot as plt, numpy as np
conf_id = confidence(Xte)
correct = (RR.predict_proba(model, Xte) >= 0.5).astype(int) == yte
ood_conf = np.concatenate([confidence(np.random.default_rng(2).random((400, 1, 64, 64), dtype='float32')), confidence(1.0 - Xte[:400])])  # OOD = noise + a flipped scan
fig, (axH, axC) = plt.subplots(1, 2, figsize=(11, 4))
# LEFT — confidence histograms: ID-correct, ID-wrong, OOD (noise + flipped scan)
axH.hist(conf_id[correct], bins=20, alpha=0.6, color='C0', label='ID correct')
axH.hist(conf_id[~correct], bins=20, alpha=0.6, color='C3', label='ID wrong')
axH.hist(ood_conf, bins=20, alpha=0.6, color='0.5', label='OOD (noise + flipped)')
axH.set(xlabel='max-softmax confidence', ylabel='count', title='Confidence cannot separate OOD')
axH.legend()
# RIGHT — coverage vs accuracy as the abstention bar moves
taus = np.linspace(0.5, 0.99, 30)
cov = [float((conf_id >= t).mean()) for t in taus]
acc = [float(correct[conf_id >= t].mean()) if (conf_id >= t).any() else np.nan for t in taus]
axC.plot(cov, acc, marker='.', color='C0')
axC.set(xlabel='coverage (fraction answered)', ylabel='accuracy on answered',
        title='Refusing the unsure raises accuracy')
plt.tight_layout(); plt.show()
print('Left: out-of-distribution inputs pile up at HIGH confidence, overlapping the correct answers — a bar')
print('cannot separate them. Right: answering fewer raises accuracy on the rest.')

**How to read the checkpoint plot.** *Left — confidence histograms.* If confidence meant knowledge, the grey **OOD** bars would sit at the left (low confidence). Instead they pile up on the **right**, on top of the blue **correct** answers — the model is *as sure about an out-of-distribution scan as about a real X-ray*. That is why a single confidence bar cannot fence off the unfamiliar. *Right — coverage vs accuracy.* Raise the abstention bar and you **answer fewer** patients (coverage falls) but are **more often right** on the ones you do — selective prediction. The honest takeaways: confidence-based abstention helps with *borderline in-distribution* cases, but **catching out-of-distribution inputs needs its own signal** (see Stretch), and refusing to answer is often the safest output.

### Discussion
- A model is **99% confident** on an upside-down scan. Whose failure is that — the model, the data pipeline, or the deployer who trusted the number?
- 'The model is 90% confident' — is that a safe thing to show a clinician? What would make it safe?
- Calibration (Module 1) said probabilities can be dishonest; abstention is the response — when is **refusing to answer** the most responsible output?

### Stretch — a real OOD detector
Max-softmax can't separate OOD. Pull the penultimate-layer embedding (ResNet-18 just before the final layer) for the training set, then score each test image by its distance to the training feature mean. Plot that novelty score for in-distribution vs noise — it separates them where confidence did not. (This is the idea behind Mahalanobis / energy-based OOD detection.)

### The actual fix — a feature-space OOD detector

A confidence bar couldn't fence OOD (an out-of-distribution scan can be *as confident as a real X-ray*). The remedy is a **separate signal**: how far an image's *internal features* sit from the training data. Embed each image with the network's **penultimate layer** (the 512-d vector before the final classifier — the same "frozen encoder" Module 3a fuses), then score **novelty** as its **Mahalanobis distance** to the training feature centre (a distance that accounts for how the features normally vary). Watch it cleanly flag in feature space the OOD a confidence bar could not.

In [ ]:
# WORKED — the fix confidence couldn't give you: a feature-distance OOD detector.
%matplotlib inline
import torch, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
dev = next(model.parameters()).device
feat_net = torch.nn.Sequential(*list(model.children())[:-1]).eval()   # ResNet-18 minus the final classifier
def embed(X):                                                          # 512-d penultimate features (batched)
    out = []
    with torch.inference_mode():
        for i in range(0, len(X), 256):
            z = feat_net(torch.tensor(np.asarray(X[i:i + 256]), dtype=torch.float32).to(dev))
            out.append(z.flatten(1).cpu().numpy())
    return np.concatenate(out)

Ztr = embed(Xtr)                                                       # the "normal" training features
mu = Ztr.mean(0)
icov = np.linalg.inv(np.cov(Ztr.T) + 1e-3 * np.eye(Ztr.shape[1]))      # inverse covariance (regularised)
def novelty(X):                                                        # Mahalanobis distance to the centre
    d = embed(X) - mu
    return np.sqrt(((d @ icov) * d).sum(axis=1))   # Mahalanobis distance

noise = np.random.default_rng(4).random((400, 1, 64, 64), dtype='float32')
labels = np.r_[np.zeros(len(Xte)), np.ones(len(noise))]               # 1 = OOD
auc = roc_auc_score(labels, np.r_[novelty(Xte), novelty(noise)])
print(f'feature-distance (Mahalanobis) OOD-detector AUROC on noise = {auc:.2f}')
print('Max-softmax could not reliably separate these OOD inputs from real X-rays; this feature-space signal flags OOD')
print('cleanly instead -> the real guardrail is a feature-space novelty detector, not the softmax.')
plt.hist(novelty(Xte), bins=20, alpha=0.6, label='ID (real X-rays)')
plt.hist(novelty(noise), bins=20, alpha=0.6, label='OOD noise')
plt.xlabel('Mahalanobis novelty'); plt.ylabel('count')
plt.title('A feature-space detector flags OOD the softmax could not'); plt.legend(); plt.show()

### Stretch (thought exercise) — SimCLR: the contrastive loss behind CLIP

You trained the CNN *with* labels. **Self-supervised contrastive learning** asks: can it learn useful features from *unlabeled* images? SimCLR's trick: make **two random augmentations of
the same image** a *positive pair*, and treat every other image in the batch as a *negative*. The encoder learns to place the two views of one X-ray close together and everything else
apart — capturing what is *invariant* (the anatomy) and discarding the augmentation noise.

This is a **thought exercise**: you write the loss and watch it, but this model + this little dataset are far too small for SimCLR to actually pay off (it needs a large batch = many
negatives, lots of data, and many epochs — really a GPU). Expect the linear-probe gain over a random network to be small and noisy. That *is* the lesson: a correct method still needs
scale. You will meet the **same loss again in Module 3a as CLIP**, where it is applied across two *modalities* (image and tabular) of one patient instead of two crops of one image — and
there, on data built to be learnable, it visibly aligns them.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import radiograph_reader as RR
torch.manual_seed(0)

# real images if cached, else the offline smoke fixture (upsampled to 64 so ResNet-18 is happy)
try:    Xs, ys = RR.load_pneumonia('train', size=64)
except Exception: Xs, ys = RR.load_pneumonia(smoke=True)
Xs = F.interpolate(torch.tensor(Xs), size=64, mode='bilinear', align_corners=False).numpy()
SUB = min(1200, len(Xs)); Xs, ys = Xs[:SUB], ys[:SUB]      # subset keeps the CPU thought-exercise quick

def augment(x):                       # two-view augmentations: random crop + brightness + noise
    B, pad = x.shape[0], 8            # (no horizontal flip -- chest-X-ray laterality is meaningful)
    xp = F.pad(x, (pad,) * 4, mode='reflect'); out = torch.empty_like(x)
    for i in range(B):
        t = int(torch.randint(0, 2 * pad + 1, (1,))); l = int(torch.randint(0, 2 * pad + 1, (1,)))
        out[i] = xp[i, :, t:t + 64, l:l + 64]
    return (out * (0.8 + 0.4 * torch.rand(B, 1, 1, 1)) + 0.05 * torch.randn_like(out)).clamp(0, 1)

backbone = RR.build_model(pretrained=False); backbone.fc = nn.Identity()   # M2's CNN trunk -> 512-d
proj = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 128))  # projection head
print('encoder = M2 ResNet-18 trunk (512-d) + projection head (128-d) | images:', Xs.shape)

In [ ]:
# WORKED -- the SimCLR NT-Xent loss.
# z has 2B rows: rows 0..B-1 are view-A of each image, rows B..2B-1 are view-B.
# The positive for row i is its OTHER view (row i+B, wrapping around); all other rows are negatives.
def nt_xent(z, temperature=0.5):
    z   = F.normalize(z, dim=1)                 # unit vectors, so a dot product = cosine similarity
    sim = z @ z.t() / temperature               # (2B, 2B) similarity matrix
    sim.fill_diagonal_(float('-inf'))           # (a) a view is never its own positive
    targets = torch.arange(z.shape[0]).roll(z.shape[0] // 2)   # (b) positive of i is i+B (wraps)
    return F.cross_entropy(sim, targets)

In [ ]:
opt = torch.optim.Adam(list(backbone.parameters()) + list(proj.parameters()), lr=1e-3)
Xt, BATCH, EPOCHS = torch.tensor(Xs), 128, 10          # bump EPOCHS / dataset up only if you have a GPU
for ep in range(EPOCHS):
    perm = torch.randperm(len(Xt)); tot = k = 0
    for i in range(0, len(Xt), BATCH):
        xb = Xt[perm[i:i + BATCH]]
        z  = proj(backbone(torch.cat([augment(xb), augment(xb)])))   # (2B, 128)
        loss = nt_xent(z); opt.zero_grad(); loss.backward(); opt.step(); tot += float(loss); k += 1
    print(f'epoch {ep + 1:2d}/{EPOCHS}  nt_xent = {tot / k:.3f}')     # noisy; should drift down a little
print('It moves, but slowly -- too few negatives at this scale. The loss is right; the scale is wrong.')

In [ ]:
from sklearn.linear_model import LogisticRegression
from common import metrics as M
import numpy as np
tr = np.arange(len(Xs)) < int(0.7 * len(Xs)); te = ~tr
def feats(net, A):
    net.eval()
    with torch.inference_mode(): return net(torch.tensor(A)).cpu().numpy()
def probe(net):   # train a LINEAR classifier on FROZEN features -> how good is the representation?
    clf = LogisticRegression(max_iter=1000).fit(feats(net, Xs[tr]), ys[tr])
    return M.auroc(ys[te], clf.predict_proba(feats(net, Xs[te]))[:, 1])
rand = RR.build_model(pretrained=False); rand.fc = nn.Identity()
print(f'linear-probe AUROC   SimCLR={probe(backbone):.3f}   random-init={probe(rand):.3f}')
print('Read it HONESTLY: SimCLR used NO labels. At this scale the gap over random is small and noisy')
print('(few test images, few negatives, few epochs) -- exactly why contrastive learning needs scale.')

**What you built.** A correct contrastive loss that learns image features with *no labels*, by pulling two views of one X-ray together and pushing other images apart. The loss is right;
this toy scale is wrong, so the payoff is muted — that honest gap is the point. **The same loss returns in Module 3a as CLIP**, where the positive pair is a patient's *image and tabular*
views and, on data built to be learnable, it visibly aligns the two modalities. One idea, two uses.